In [47]:
import os
import json
import math
import random
import string
import numpy as np
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, login

# Configurations and hyperparameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN:
    login(token=HF_TOKEN)

# Paths for Phase 1 Checkpoint & Tokenizer
TOKENIZER_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json"
CHECKPOINT_PATH = "/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt"

# Phase 2 Hyperparameters
MAX_SEQ_LEN = 512
STRIDE = 128
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4      # adapter (LoRMA) params
HEAD_LR = 1e-3            # classifier head
EPOCHS = 10
LORA_RANK = 32
LORA_ALPHA = 64.0
USE_VAL = True           # True: train fold1, validate on fold2.  False: final run on folds 1+2
FT_MODE = "full"        # "lorma" | "full" | "top_k"
TOP_K = 4               # used by "top_k": unfreeze the last K blocks
BACKBONE_LR = 5e-5      # used by "full" and "top_k"

# 12 Official Labels
LABEL_NAMES = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE",
    "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
# BIO Scheme (25 tags total)
ID2LABEL = {0: "O"}
LABEL2ID = {"O": 0}
for idx, lbl in enumerate(LABEL_NAMES):
    ID2LABEL[1 + 2 * idx] = f"B-{lbl}"
    LABEL2ID[f"B-{lbl}"] = 1 + 2 * idx
    ID2LABEL[2 + 2 * idx] = f"I-{lbl}"
    LABEL2ID[f"I-{lbl}"] = 2 + 2 * idx
NUM_CLASSES = len(ID2LABEL)
IGNORE_INDEX = -100
print(f"Device: {DEVICE} | Classes: {NUM_CLASSES} | USE_VAL={USE_VAL}")

Device: cuda | Classes: 25 | USE_VAL=True


In [38]:
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
END_OF_WORD = "</w>"

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []
        self.merge_rank = {}
        self.itos = []
        self.stoi = {}

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(vocab_size=data["vocab_size"])
        tok.itos = data["itos"]
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.merges = [tuple(p) for p in data["merges"]]
        tok.merge_rank = {p: i for i, p in enumerate(tok.merges)}
        return tok

    def _word_to_symbols(self, word):
        return list(word.encode("utf-8").decode("latin-1")) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        symbols = self._word_to_symbols(word)
        while len(symbols) > 1:
            pairs_here = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            candidates = [(self.merge_rank[p], idx) for idx, p in enumerate(pairs_here) if p in self.merge_rank]
            if not candidates:
                break
            _, idx = min(candidates)
            merged = symbols[idx] + symbols[idx + 1]
            symbols = symbols[:idx] + [merged] + symbols[idx + 2:]
        return symbols

    def encode_with_exact_offsets(self, text: str) -> Tuple[List[int], List[Tuple[int, int]]]:
        """
        Calculates EXACT character-level [start, end) offsets for every single subword token.
        """
        tokens = []
        offsets = []
        i = 0
        n = len(text)

        while i < n:
            while i < n and text[i].isspace():
                i += 1
            if i >= n:
                break
            word_start = i
            while i < n and not text[i].isspace():
                i += 1
            word_end = i
            word = text[word_start:word_end]

            subwords = self._apply_merges_to_word(word)
            curr_pos = word_start

            for sym in subwords:
                clean_sym = sym.replace(END_OF_WORD, "")
                try:
                    sym_chars = clean_sym.encode("latin-1").decode("utf-8", errors="replace")
                except Exception:
                    sym_chars = clean_sym

                sub_len = len(sym_chars)
                sub_start = curr_pos
                sub_end = min(word_end, curr_pos + sub_len)
                curr_pos = sub_end

                t_id = self.stoi.get(sym, self.stoi.get("<unk>", 1))
                tokens.append(t_id)
                offsets.append((sub_start, sub_end))

        return tokens, offsets

# Load tokenizer
tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
print(f"Loaded tokenizer | Vocab size: {tokenizer.vocab_size}")

Loaded tokenizer | Vocab size: 8000


In [39]:
import zipfile
import os

source_dir = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'
target_pt = '/kaggle/working/decoder_lm.pt'

# Re-package the folder into the single PyTorch archive file
with zipfile.ZipFile(target_pt, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            full_path = os.path.join(root, file)
            # relative path inside the archive
            rel_path = os.path.relpath(full_path, source_dir)
            zf.write(full_path, arcname=rel_path)

print(f"✓ Created valid checkpoint file: {target_pt} ({os.path.getsize(target_pt)/1e6:.1f} MB)")

# Set CHECKPOINT_PATH to this working file
CHECKPOINT_PATH = target_pt

✓ Created valid checkpoint file: /kaggle/working/decoder_lm.pt (421.1 MB)


In [40]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads: int):
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads))
    else:
        closest_pow2 = 2 ** math.floor(math.log2(n_heads))
        slopes_a = get_slopes_power_of_2(closest_pow2)
        slopes_b = get_slopes_power_of_2(2 * closest_pow2)[0::2][:n_heads - closest_pow2]
        return torch.tensor(slopes_a + slopes_b)

class LORMALinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.original_linear = original_linear
        self.rank = rank
        self.scaling = alpha / rank
        self.original_linear.weight.requires_grad = False

        in_f = original_linear.in_features
        out_f = original_linear.out_features
        self.lorma_down = nn.Linear(in_f, rank, bias=False)
        self.lorma_up = nn.Linear(rank, out_f, bias=False)

        nn.init.kaiming_uniform_(self.lorma_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.original_linear(x) + self.lorma_up(self.lorma_down(x)) * self.scaling

class ALiBiBidirectionalAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer("slopes", slopes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        # Symmetric bidirectional distance
        pos = torch.arange(S, device=x.device)
        dist = -torch.abs(pos.unsqueeze(0) - pos.unsqueeze(1)).float()
        alibi_bias = self.slopes * dist.unsqueeze(0)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=alibi_bias,
            dropout_p=self.dropout if self.training else 0.0
        )
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.out_proj(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = ALiBiBidirectionalAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Arch4LNERModel(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 640, n_layers: int = 16,
                 n_heads: int = 8, d_ff: int = 2560, num_classes: int = NUM_CLASSES,
                 dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.drop(self.tok_emb(input_ids))
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        return self.classifier(x)

    def apply_lorma(self, rank: int = 16, alpha: float = 32.0, device=DEVICE):
        for param in self.parameters():
            param.requires_grad = False

        for block in self.blocks:
            block.attn.q_proj = LORMALinear(block.attn.q_proj, rank=rank, alpha=alpha).to(device)
            block.attn.v_proj = LORMALinear(block.attn.v_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.gate_proj = LORMALinear(block.ffn.gate_proj, rank=rank, alpha=alpha).to(device)
            block.ffn.up_proj = LORMALinear(block.ffn.up_proj, rank=rank, alpha=alpha).to(device)

        for param in self.classifier.parameters():
            param.requires_grad = True

# ---- NEW: adapters on every linear layer ----
def apply_lorma_all(model, rank, alpha, device):
    for p in model.parameters():
        p.requires_grad = False
    for b in model.blocks:
        for n in ["q_proj", "k_proj", "v_proj", "out_proj"]:
            setattr(b.attn, n, LORMALinear(getattr(b.attn, n), rank, alpha).to(device))
        for n in ["gate_proj", "up_proj", "down_proj"]:
            setattr(b.ffn, n, LORMALinear(getattr(b.ffn, n), rank, alpha).to(device))
    for p in model.classifier.parameters():
        p.requires_grad = True
    for p in model.norm_f.parameters():
        p.requires_grad = True

# Instantiate & Load Checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
d_model_ckpt = state_dict["tok_emb.weight"].shape[1]
vocab_size_ckpt = state_dict["tok_emb.weight"].shape[0]
block_indices = [int(k.split(".")[1]) for k in state_dict.keys() if k.startswith("blocks.") and k.split(".")[1].isdigit()]
n_layers_ckpt = max(block_indices) + 1 if block_indices else 16
n_heads = 8 if d_model_ckpt == 640 else 12

model = Arch4LNERModel(vocab_size=vocab_size_ckpt, d_model=d_model_ckpt, n_layers=n_layers_ckpt, n_heads=n_heads).to(DEVICE)
backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith("lm_head") and not k.startswith("head")}
missing, unexpected = model.load_state_dict(backbone_dict, strict=False)
print("MISSING   :", len(missing), missing[:8])
print("UNEXPECTED:", len(unexpected), unexpected[:8])
assert all(m.startswith("classifier.") or m.endswith(".slopes") for m in missing), \
    "Backbone weights did NOT load - fix key names before continuing!"

def setup_trainable(model, mode):
    if mode == "lorma":
        apply_lorma_all(model, LORA_RANK, LORA_ALPHA, DEVICE)
    elif mode == "full":
        for p in model.parameters(): p.requires_grad = True
    elif mode == "top_k":
        for p in model.parameters(): p.requires_grad = False
        for b in model.blocks[-TOP_K:]:
            for p in b.parameters(): p.requires_grad = True
        for p in model.norm_f.parameters(): p.requires_grad = True
        for p in model.classifier.parameters(): p.requires_grad = True
    model.to(DEVICE)

setup_trainable(model, FT_MODE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Params: {total_params:,} | Trainable Params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

MISSING   : 6 ['classifier.0.weight', 'classifier.0.bias', 'classifier.1.weight', 'classifier.1.bias', 'classifier.4.weight', 'classifier.4.bias']
UNEXPECTED: 0 []
Total Params: 105,866,521 | Trainable Params: 105,866,521 (100.00%)


In [17]:
def download_phase2_file(filename: str) -> str:
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

print("Downloading dataset files...")
f1_text_path = download_phase2_file("Phase_2_LNER/Train/fold1-text.json")
f1_gold_path = download_phase2_file("Phase_2_LNER/Train/fold1-gold.json")
f2_text_path = download_phase2_file("Phase_2_LNER/Train/fold2-text.json")
f2_gold_path = download_phase2_file("Phase_2_LNER/Train/fold2-gold.json")
f3_text_path = download_phase2_file("Phase_2_LNER/Test/fold3-text.json")

with open(f1_text_path, "r", encoding="utf-8") as f: train_texts = json.load(f)
with open(f1_gold_path, "r", encoding="utf-8") as f: train_golds = json.load(f)
with open(f2_text_path, "r", encoding="utf-8") as f: val_texts = json.load(f)
with open(f2_gold_path, "r", encoding="utf-8") as f: val_golds = json.load(f)
with open(f3_text_path, "r", encoding="utf-8") as f: test_texts = json.load(f)

# ---- NEW: held-out validation switch ----
if USE_VAL:
    tr_texts, tr_golds = train_texts, train_golds
else:
    tr_texts = {**train_texts, **val_texts}
    tr_golds = {**train_golds, **val_golds}
print(f"Training docs: {len(tr_texts)} | USE_VAL={USE_VAL}")

# ---- NEW: windows where the last one is always full-length (no tiny padded tail) ----
def window_starts(n, max_len=MAX_SEQ_LEN, stride=STRIDE):
    if n <= max_len:
        return [0]
    starts = list(range(0, n - max_len + 1, max_len - stride))
    if starts[-1] + max_len < n:
        starts.append(n - max_len)
    return starts

class ExactLNERDataset(Dataset):
    def __init__(self, docs: Dict[str, str], golds: Dict[str, List[Any]], tokenizer: ByteBPETokenizer,
                 max_len: int = MAX_SEQ_LEN, stride: int = STRIDE):
        self.samples = []
        for doc_id, text in docs.items():
            token_ids, offsets = tokenizer.encode_with_exact_offsets(text)

            token_labels = [0] * len(token_ids)
            if doc_id in golds:
                for annot in golds[doc_id]:
                    g_start, g_end, g_lbl = annot[0], annot[1], annot[2]
                    first_matched = False
                    for idx, (t_s, t_e) in enumerate(offsets):
                        if max(t_s, g_start) < min(t_e, g_end):
                            if not first_matched:
                                token_labels[idx] = LABEL2ID.get(f"B-{g_lbl}", 0)
                                first_matched = True
                            else:
                                token_labels[idx] = LABEL2ID.get(f"I-{g_lbl}", 0)

            for start_idx in window_starts(len(token_ids), max_len, stride):
                end_idx = min(len(token_ids), start_idx + max_len)
                c_ids = token_ids[start_idx:end_idx]
                c_lbls = token_labels[start_idx:end_idx]

                pad_len = max_len - len(c_ids)
                padded_ids = c_ids + [0] * pad_len
                padded_lbls = c_lbls + [IGNORE_INDEX] * pad_len

                self.samples.append({
                    "input_ids": torch.tensor(padded_ids, dtype=torch.long),
                    "labels": torch.tensor(padded_lbls, dtype=torch.long)
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

train_ds = ExactLNERDataset(tr_texts, tr_golds, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Total training sliding windows: {len(train_ds)}")

Training docs: 35 | USE_VAL=True
Total training sliding windows: 652


In [31]:
# ---- boundary cleanup: strip punctuation (incl. curly quotes/dashes) at span edges ----
EXTRA = "\u2018\u2019\u201c\u201d\u2013\u2014\u2022\u00b7\u2026"
STRIP_LEAD = " \t\n\r" + string.punctuation + EXTRA
STRIP_TRAIL = " \t\n\r" + string.punctuation + EXTRA

def clean_span(text: str, start: int, end: int, label: str):
    while start < end and text[start] in STRIP_LEAD:
        start += 1
    while end > start and text[end - 1] in STRIP_TRAIL:
        end -= 1
    return start, end, label

def decode_spans_exact(doc_text: str, pred_ids: List[int], offsets: List[Tuple[int, int]]) -> List[List[Any]]:
    spans, cur_l, cs, ce = [], None, -1, -1

    def flush():
        if cur_l is not None:
            s, e, l = clean_span(doc_text, cs, ce, cur_l)
            if e > s:
                spans.append([s, e, l])

    for t_id, (s_c, e_c) in zip(pred_ids, offsets):
        tag = ID2LABEL.get(int(t_id), "O")
        if tag.startswith("B-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        elif tag.startswith("I-") and cur_l == tag[2:]:
            ce = e_c
        elif tag.startswith("I-"):
            flush(); cur_l, cs, ce = tag[2:], s_c, e_c
        else:
            flush(); cur_l = None
    flush()

    seen, out = set(), []
    for s in spans:
        if tuple(s) not in seen:
            seen.add(tuple(s)); out.append(s)
    return out

# ---- BIO-constrained Viterbi (with optional continuation bonus) ----
NEG = -1e9
ALLOWED = np.ones((NUM_CLASSES, NUM_CLASSES), dtype=bool)     # [prev, cur]
for cur in range(NUM_CLASSES):
    if ID2LABEL[cur].startswith("I-"):
        for prev in range(NUM_CLASSES):
            ALLOWED[prev, cur] = (ID2LABEL[prev] != "O") and (ID2LABEL[prev][2:] == ID2LABEL[cur][2:])
START_OK = np.array([not ID2LABEL[c].startswith("I-") for c in range(NUM_CLASSES)])

def make_trans(cont_bonus=0.0):
    t = np.where(ALLOWED, 0.0, NEG).astype(np.float64)
    for cur in range(NUM_CLASSES):
        if ID2LABEL[cur].startswith("I-"):
            for prev in range(NUM_CLASSES):
                if ALLOWED[prev, cur]:
                    t[prev, cur] += cont_bonus
    return t

def viterbi(logp, trans):
    T, C = logp.shape
    score = logp[0].copy(); score[~START_OK] = NEG
    back = np.zeros((T, C), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = [int(score.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

def merge_spans(text, spans, max_gap=0):
    out = []
    for s, e, l in sorted(spans):
        if out and out[-1][2] == l and 0 <= s - out[-1][1] <= max_gap and text[out[-1][1]:s].strip() == "":
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e, l])
    return out

# ---- per-document log-probs, averaged over overlapping windows; padded exactly like training ----
@torch.no_grad()
def doc_logp(text):
    model.eval()
    ids, offs = tokenizer.encode_with_exact_offsets(text)
    if not ids:
        return None, None
    acc = torch.zeros(len(ids), NUM_CLASSES); cnt = torch.zeros(len(ids), 1)
    for s in window_starts(len(ids)):
        e = min(len(ids), s + MAX_SEQ_LEN)
        chunk = ids[s:e]
        x = torch.tensor([chunk + [0] * (MAX_SEQ_LEN - len(chunk))], dtype=torch.long, device=DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            lp = F.log_softmax(model(x)[0].float(), dim=-1)[: len(chunk)].cpu()
        acc[s:e] += lp; cnt[s:e] += 1
    return (acc / cnt).numpy(), offs

def decode_cached(cache, texts, o_bias=0.0, cont_bonus=0.0, max_gap=0):
    trans, out = make_trans(cont_bonus), {}
    for d, (lp, offs) in cache.items():
        if lp is None:
            out[d] = []
            continue
        lp = lp.astype(np.float64).copy()
        lp[:, 0] += o_bias
        sp = decode_spans_exact(texts[d], viterbi(lp, trans), offs)
        out[d] = merge_spans(texts[d], sp, max_gap) if max_gap > 0 else sp
    return out

def strict_macro_f1(pred, gold, labels=LABEL_NAMES):
    f1s = []
    for lbl in labels:
        tp = fp = fn = 0
        for d in gold:
            g = {(a[0], a[1]) for a in gold[d] if a[2] == lbl}
            p = {(a[0], a[1]) for a in pred.get(d, []) if a[2] == lbl}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s), dict(zip(labels, f1s))

In [48]:
# ---- class weights: O = 1, rare tags up to 20x ----
counts = torch.zeros(NUM_CLASSES)
for s in train_ds.samples:
    l = s["labels"]; l = l[l != IGNORE_INDEX]
    counts += torch.bincount(l, minlength=NUM_CLASSES).float()
w = (counts.sum() / counts.clamp(min=1)) ** 0.5
w = (w / w[0]).clamp(max=20.0)
print("O fraction:", round((counts[0] / counts.sum()).item(), 4))
criterion = nn.CrossEntropyLoss(weight=w.to(DEVICE), ignore_index=IGNORE_INDEX)

# ---- optimizer with a separate (higher) LR for the head ----
head_params = list(model.classifier.parameters()); head_ids = {id(p) for p in head_params}
emb_params = [p for p in model.tok_emb.parameters() if p.requires_grad]; emb_ids = {id(p) for p in emb_params}
rest = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids and id(p) not in emb_ids]
groups = [{"params": head_params, "lr": HEAD_LR},
          {"params": rest, "lr": BACKBONE_LR if FT_MODE in ("full", "top_k") else LEARNING_RATE}]
if emb_params:
    groups.append({"params": emb_params, "lr": 2e-5})
optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.05 * total_steps))
def lr_lambda(s):
    if s < warmup:
        return (s + 1) / warmup
    prog = (s - warmup) / max(1, total_steps - warmup)
    return 0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Training {EPOCHS} epochs | USE_VAL={USE_VAL}")
best_f1, best_state, best_epoch = -1.0, None, 0

for epoch in range(EPOCHS):
    model.train()
    running, n_log = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(input_ids)
        loss = criterion(logits.float().view(-1, NUM_CLASSES), labels.view(-1))

        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        running += loss.item(); n_log += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch + 1}/{EPOCHS}] | Step [{step + 1}/{len(train_loader)}] | Loss: {running / n_log:.4f}")
            running, n_log = 0.0, 0

    if USE_VAL and ((epoch + 1) % 2 == 0 or epoch + 1 == EPOCHS):
        vc = {d: doc_logp(t) for d, t in val_texts.items()}
        f1, per = strict_macro_f1(decode_cached(vc, val_texts, 1.5), val_golds)
        print(f"==> Epoch {epoch + 1} | VAL strict macro-F1 (o_bias=1.5): {f1:.4f}")
        print({k: round(v, 3) for k, v in per.items()})
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            best_state = {k: v.detach().clone() for k, v in model.named_parameters() if v.requires_grad}

if USE_VAL and best_state is not None:
    model.load_state_dict(best_state, strict=False)
    print(f"Restored best epoch {best_epoch} (val F1 {best_f1:.4f})")
print("✓ Training finished!")

O fraction: 0.9023
Training 10 epochs | USE_VAL=True
Epoch [1/10] | Step [100/163] | Loss: 0.0202
Epoch [2/10] | Step [100/163] | Loss: 0.0198
==> Epoch 2 | VAL strict macro-F1 (o_bias=1.5): 0.3203
{'APP': 0.067, 'RESP': 0.075, 'A.COUNSEL': 0.196, 'R.COUNSEL': 0.167, 'JUDGE': 0.304, 'WIT': 0.145, 'AUTH': 0.248, 'COURT': 0.79, 'STAT': 0.544, 'PREC': 0.169, 'DATE': 0.852, 'CASENO': 0.286}
Epoch [3/10] | Step [100/163] | Loss: 0.0136
Epoch [4/10] | Step [100/163] | Loss: 0.0107
==> Epoch 4 | VAL strict macro-F1 (o_bias=1.5): 0.3432
{'APP': 0.085, 'RESP': 0.086, 'A.COUNSEL': 0.261, 'R.COUNSEL': 0.152, 'JUDGE': 0.355, 'WIT': 0.14, 'AUTH': 0.281, 'COURT': 0.777, 'STAT': 0.568, 'PREC': 0.201, 'DATE': 0.858, 'CASENO': 0.354}
Epoch [5/10] | Step [100/163] | Loss: 0.0065
Epoch [6/10] | Step [100/163] | Loss: 0.0044
==> Epoch 6 | VAL strict macro-F1 (o_bias=1.5): 0.3334
{'APP': 0.088, 'RESP': 0.076, 'A.COUNSEL': 0.21, 'R.COUNSEL': 0.158, 'JUDGE': 0.324, 'WIT': 0.148, 'AUTH': 0.23, 'COURT': 0.79, 

In [49]:
test_cache = {d: doc_logp(t) for d, t in test_texts.items()}
pickle.dump(test_cache, open("/kaggle/working/test_full.pkl", "wb"))

In [35]:
val_cache = {d: doc_logp(t) for d, t in val_texts.items()}

# stage 1: O bias x continuation bonus
best = (-1.0, None)
for ob in [0.0, 1.0, 1.5, 2.0, 3.0]:
    for cb in [0.0, 0.5, 1.0, 2.0]:
        f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, ob, cb), val_golds)
        print(f"o_bias={ob} cont_bonus={cb} -> {f:.4f}")
        if f > best[0]:
            best = (f, (ob, cb))
BEST_O_BIAS, BEST_CONT = best[1]
print("best (o_bias, cont_bonus):", best)

# stage 2: merge gap on top of the best setting
BEST_MAX_GAP, best_g = 0, best[0]
for g in [1, 2, 3]:
    f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT, g), val_golds)
    print(f"max_gap={g} -> {f:.4f}")
    if f > best_g:
        best_g, BEST_MAX_GAP = f, g
print(f"\nFINAL SETTINGS: o_bias={BEST_O_BIAS} cont_bonus={BEST_CONT} max_gap={BEST_MAX_GAP} | val F1={best_g:.4f}")

final_val = decode_cached(val_cache, val_texts, BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP)
final_val = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in final_val.items()}
evaluate_predictions_strictly(val_golds, final_val)

o_bias=0.0 cont_bonus=0.0 -> 0.3550
o_bias=0.0 cont_bonus=0.5 -> 0.3641
o_bias=0.0 cont_bonus=1.0 -> 0.3706
o_bias=0.0 cont_bonus=2.0 -> 0.3768
o_bias=1.0 cont_bonus=0.0 -> 0.3529
o_bias=1.0 cont_bonus=0.5 -> 0.3624
o_bias=1.0 cont_bonus=1.0 -> 0.3736
o_bias=1.0 cont_bonus=2.0 -> 0.3804
o_bias=1.5 cont_bonus=0.0 -> 0.3443
o_bias=1.5 cont_bonus=0.5 -> 0.3604
o_bias=1.5 cont_bonus=1.0 -> 0.3704
o_bias=1.5 cont_bonus=2.0 -> 0.3847
o_bias=2.0 cont_bonus=0.0 -> 0.3385
o_bias=2.0 cont_bonus=0.5 -> 0.3513
o_bias=2.0 cont_bonus=1.0 -> 0.3660
o_bias=2.0 cont_bonus=2.0 -> 0.3840
o_bias=3.0 cont_bonus=0.0 -> 0.3184
o_bias=3.0 cont_bonus=0.5 -> 0.3344
o_bias=3.0 cont_bonus=1.0 -> 0.3492
o_bias=3.0 cont_bonus=2.0 -> 0.3780
best (o_bias, cont_bonus): (0.3846791718042546, (1.5, 2.0))
max_gap=1 -> 0.3889
max_gap=2 -> 0.3889
max_gap=3 -> 0.3889

FINAL SETTINGS: o_bias=1.5 cont_bonus=2.0 max_gap=1 | val F1=0.3889


NameError: name 'evaluate_predictions_strictly' is not defined

In [24]:
best = (-1.0, None)
for ob in [1.0, 1.5, 2.0, 2.5, 3.0]:
    for cb in [2.0, 3.0, 4.0, 6.0]:
        f, _ = strict_macro_f1(decode_cached(val_cache, val_texts, ob, cb), val_golds)
        print(f"o_bias={ob} cont_bonus={cb} -> {f:.4f}")
        if f > best[0]:
            best = (f, (ob, cb))
BEST_O_BIAS, BEST_CONT = best[1]
print("best:", best)

o_bias=1.0 cont_bonus=2.0 -> 0.3821
o_bias=1.0 cont_bonus=3.0 -> 0.3745
o_bias=1.0 cont_bonus=4.0 -> 0.3560
o_bias=1.0 cont_bonus=6.0 -> 0.3142
o_bias=1.5 cont_bonus=2.0 -> 0.3906
o_bias=1.5 cont_bonus=3.0 -> 0.3837
o_bias=1.5 cont_bonus=4.0 -> 0.3672
o_bias=1.5 cont_bonus=6.0 -> 0.3264
o_bias=2.0 cont_bonus=2.0 -> 0.3926
o_bias=2.0 cont_bonus=3.0 -> 0.3916
o_bias=2.0 cont_bonus=4.0 -> 0.3800
o_bias=2.0 cont_bonus=6.0 -> 0.3416
o_bias=2.5 cont_bonus=2.0 -> 0.3932
o_bias=2.5 cont_bonus=3.0 -> 0.4007
o_bias=2.5 cont_bonus=4.0 -> 0.3898
o_bias=2.5 cont_bonus=6.0 -> 0.3555
o_bias=3.0 cont_bonus=2.0 -> 0.3890
o_bias=3.0 cont_bonus=3.0 -> 0.3999
o_bias=3.0 cont_bonus=4.0 -> 0.3988
o_bias=3.0 cont_bonus=6.0 -> 0.3654
best: (0.4007368190757871, (2.5, 3.0))


In [50]:
import pickle, os, json, zipfile

def ensemble_caches(caches, weights):
    out = {}
    for d in caches[0]:
        if caches[0][d][0] is None:
            out[d] = (None, None); continue
        out[d] = (sum(w * c[d][0] for w, c in zip(weights, caches)), caches[0][d][1])
    return out

lorma = pickle.load(open("/kaggle/working/test_lorma.pkl", "rb"))
full  = pickle.load(open("/kaggle/working/test_full.pkl", "rb"))
ens = ensemble_caches([lorma, full], weights=[0.6, 0.4])   # LoRMA gets more weight, it's the stronger model

preds = decode_cached(ens, test_texts, 2.5, 3.0)
preds = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in preds.items()}

with open("/kaggle/working/prediction.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(preds)); f.flush(); os.fsync(f.fileno())
if os.path.exists("/kaggle/working/submission.zip"): os.remove("/kaggle/working/submission.zip")
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("✓ ensemble submission ready")

✓ ensemble submission ready


In [52]:
val_ens = ensemble_caches([load_cache("/kaggle/working/val_lorma.pkl"),
                           load_cache("/kaggle/working/val_full.pkl")], [0.7, 0.3])
for ob, cb in [(2.0, 2.0), (2.5, 3.0), (3.0, 3.0), (2.5, 4.0)]:
    print(ob, cb, round(strict_macro_f1(decode_cached(val_ens, val_texts, ob, cb), val_golds)[0], 4))

2.0 2.0 0.3964
2.5 3.0 0.4036
3.0 3.0 0.4044
2.5 4.0 0.4059


In [36]:
# BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP come from Cell 7 (or set them by hand, e.g. 2.0, 0.5, 0)
test_cache = {d: doc_logp(t) for d, t in test_texts.items()}
test_predictions = decode_cached(test_cache, test_texts, BEST_O_BIAS, BEST_CONT, BEST_MAX_GAP)
test_predictions = {d: [[int(a[0]), int(a[1]), str(a[2])] for a in sp] for d, sp in test_predictions.items()}

output_json = "/kaggle/working/prediction.json"
output_zip = "/kaggle/working/submission.zip"
with open(output_json, "w", encoding="utf-8") as f:
    f.write(json.dumps(test_predictions))
    f.flush()
    os.fsync(f.fileno())
if os.path.exists(output_zip):
    os.remove(output_zip)
with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_json, arcname="prediction.json")
print(f"\n✓ Submission zip created: {output_zip} ({os.path.getsize(output_zip):,} bytes)")

with zipfile.ZipFile(output_zip, "r") as zf:
    verified = json.loads(zf.read("prediction.json").decode("utf-8"))
    first_k = list(verified.keys())[0]
    print(f"✓ ZIP VERIFIED: Found {len(verified)} documents.")
    print(f"Sample prediction for doc {first_k}: {verified[first_k][:3]}")


✓ Submission zip created: /kaggle/working/submission.zip (28,382 bytes)
✓ ZIP VERIFIED: Found 35 documents.
Sample prediction for doc 1128835: [[18, 40, 'COURT'], [70, 98, 'CASENO'], [99, 110, 'APP']]


In [37]:
import pickle
def save_cache(cache, path): pickle.dump(cache, open(path, "wb"))
def load_cache(path): return pickle.load(open(path, "rb"))
save_cache(val_cache, "/kaggle/working/val_full.pkl")